In [0]:
# Step 1: Define the Genie tool — a Python callable that sends a natural-language
# question to a Databricks Genie space and polls for the completed response.
from databricks.sdk import WorkspaceClient
import time
import re
import mlflow
# Initialize a workspace client using the current notebook's authentication.
w = WorkspaceClient()

# The Genie space ID that backs employee-related questions.
SPACE_ID = "01f1a7cce8341affb459c8c51394741b"

# def genie_tool(question):
#     """Send a question to the Genie space and return the text response."""
#     # Start a new conversation in the Genie space with the user's question.
#     response = w.api_client.do(
#         "POST",
#         f"/api/2.0/genie/spaces/{SPACE_ID}/start-conversation",
#         body={
#             "content": question
#         }
#     )
@mlflow.trace
def genie_tool(question, space_id):

    response = w.api_client.do(
        "POST",
        f"/api/2.0/genie/spaces/{space_id}/start-conversation",
        body={"content": question}
    )

    conversation_id = response["conversation_id"]
    message_id = response["message_id"]

    # Poll the conversation up to 30 times (2-second intervals = 60s max wait).
    for _ in range(30):

        time.sleep(2)

        # Fetch the current status of the message.
        msg = w.api_client.do(
            "GET",
            f"/api/2.0/genie/spaces/{SPACE_ID}/conversations/{conversation_id}/messages/{message_id}"
        )

        # Once Genie has finished processing, extract the text content.
        if msg.get("status") == "COMPLETED":

            for attachment in msg.get(
                "attachments",
                []
            ):

                if "text" in attachment:

                    return attachment["text"].get(
                        "content",
                        ""
                    )

    # Timeout fallback if Genie does not respond within the polling window.
    return "No Genie response returned."

In [0]:
# Step 2: Define the HR agent — a routing function that inspects the user's
# question and dispatches to the appropriate tool(s). A single question can
# trigger multiple tools (e.g., bonus + Genie) and all results are combined.


@mlflow.trace
def hr_agent(question, genie_space_id):
    """Route an HR question to the relevant tool(s) and return a combined response."""

    responses = []
    tools_used = []

    question_lower = question.lower()

    # --- Bonus Tool ---
    # If the question mentions "bonus", extract the salary and call the
    # Unity Catalog SQL function calculate_bonus directly via spark.sql().

    if "bonus" in question_lower:

        # Extract the first integer from the question (assumed to be the salary).
        salary_match = re.findall(
            r"\d+",
            question
        )

        if salary_match:

            salary = int(
                salary_match[0]
            )

            # Call the SQL function hr_catalog.hr_core.calculate_bonus directly.
            bonus = spark.sql(
                f"SELECT hr_catalog.hr_core.calculate_bonus({salary}) AS bonus"
            ).collect()[0]["bonus"]

            tools_used.append(
                "calculate_bonus"
            )

            responses.append(
                f"Bonus for salary {salary}: {bonus}"
            )

    # --- Genie Tool ---
    # If the question contains any employee-related keywords, delegate to
    # the Genie space for a natural-language answer.

    genie_keywords = [
        "employee",
        "employees",
        "leave",
        "headcount",
        "training"
    ]

    if any(
        word in question_lower
        for word in genie_keywords
    ):

        genie_answer = genie_tool(
                        question,
                        genie_space_id
            )

        tools_used.append(
            "genie_tool"
        )

        responses.append(
            genie_answer
        )

    # Build the output: list which tools were used, then append all responses.
    output = ""

    if tools_used:

        output += (
            "Tools Used:\n"
            + "\n".join(
                f"- {tool}"
                for tool in tools_used
            )
            + "\n\n"
        )

    output += "\n\n".join(responses)

    return output

In [0]:
# Step 3: Test the agent with a bonus-only question.
# Expected: the agent detects "bonus", extracts salary 100000, calls the SQL
# function calculate_bonus, and returns the bonus amount (10000.0).

VALID_SPACE_ID = "01f1a7cce8341affb459c8c51394741b"

print(
    hr_agent(
        "What is the bonus for a salary of 100000?",
        VALID_SPACE_ID
    )
)


In [0]:
# Step 4: Test the agent with a Genie-only question.
# Expected: the agent detects the keyword "employees", delegates to genie_tool,
# and returns the Genie space's natural-language response.
print(hr_agent("How many active employees do we have?", VALID_SPACE_ID))

In [0]:
# Step 5: Test the agent with a multi-tool question that triggers BOTH tools.
# Expected: the agent calls calculate_bonus (bonus) and genie_tool (employee count),
# then combines both results into a single response.
print(hr_agent("How many active employees do we have and what is the bonus for a salary of 100000?", VALID_SPACE_ID))